# Transitsuche bei Exoplaneten

Das Notebook sucht in einem Quartal Kepler-Pixeldaten von Kepler-8 mit Box Least Squares (BLS) nach dem Transit und passt danach ein physikalisches Transitmodell mit quadratischer Randverdunkelung an sechs Monate Lichtkurven in 1-Minuten-Kadenz an. Daraus folgen Radiusverhältnis, Stoßparameter, Bahnneigung und die mittlere Dichte des Sterns, die mit dem NASA Exoplanet Archive verglichen werden.

Wie bestimmt man aus den periodischen Helligkeitseinbrüchen eines Sterns die Umlaufzeit, den Radius des Planeten und die mittlere Dichte des Sterns?

#### **Hintergrund**

Zieht ein Exoplanet von der Erde aus gesehen vor seinem Stern vorbei, verdeckt er einen kleinen Teil der Sternscheibe. Dieser periodische Helligkeitsabfall heißt Transit. Die Weltraummissionen Kepler und TESS haben so den größten Teil der bekannten Exoplaneten gefunden.

Die Form des Einbruchs enthält die Geometrie des Systems:
- **Transittiefe $\delta$:** ungefähr das Flächenverhältnis $(R_p/R_\star)^2$.
- **Transitdauer $T_{14}$:** hängt über Keplers drittes Gesetz von der Bahngröße $a/R_\star$ und vom Stoßparameter $b$ ab.
- **Ein- und Austritt (Ingress, Egress):** ihre Dauer hängt vom Radiusverhältnis und von $b$ ab; die gekrümmte Form des Bodens kommt von der Randverdunkelung des Sterns.

Um periodische Einbrüche im Rauschen zu finden, nutzt man den **Box-Least-Squares-Algorithmus (BLS)**. Er modelliert den Transit als periodische Kastenfunktion und sucht auf einem Gitter aus Perioden und Dauern die Kombination, die die Daten am besten erklärt. Für die genauen Parameter braucht man danach ein physikalisches Modell, das auch Ein- und Austritt und die Randverdunkelung beschreibt.

#### **Herleitung**

**1. Geometrische Transittiefe.** Verdeckt eine dunkle Planetenscheibe mit Radius $R_p$ eine gleichmäßig helle Sternscheibe mit Radius $R_\star$, sinkt der Fluss um
$$\delta = \frac{\Delta F}{F} = \left(\frac{R_p}{R_\star}\right)^2 = k^2 .$$
Vor einem sonnenähnlichen Stern ergibt ein jupitergroßer Planet etwa $1\,\%$, ein erdgroßer Planet etwa $84$ ppm ($8.4\cdot 10^{-5}$).

**2. Stoßparameter und Transitdauer.** Für eine Kreisbahn mit großer Halbachse $a$ und Bahnneigung $i$ ist der kleinste projizierte Abstand der Mittelpunkte, in Sternradien,
$$b = \frac{a}{R_\star}\cos i .$$
Ein Transit findet statt, wenn $b < 1 + k$. Die Dauer vom ersten bis zum vierten Kontakt ist
$$T_{14} = \frac{P}{\pi}\arcsin\left(\frac{R_\star}{a}\,\frac{\sqrt{(1+k)^2 - b^2}}{\sin i}\right) \approx \frac{P}{\pi}\,\frac{R_\star}{a}\,\sqrt{(1+k)^2 - b^2} .$$

**3. Dauer von Ein- und Austritt.** Während des Eintritts schiebt sich die Planetenscheibe über den Sternrand. Die Planetenscheibe legt dabei ihren eigenen Durchmesser zurück, schräg zum Sternrand. In der gleichen Näherung (kleines $k$, $b$ nicht nahe 1) dauert das
$$\tau \approx \frac{P}{\pi}\,\frac{R_\star}{a}\,\frac{k}{\sqrt{1 - b^2}} .$$
Für Kepler-8 b ($P = 3.52$ d, $a/R_\star \approx 6.9$, $k \approx 0.096$, $b \approx 0.72$) sind das etwa 33 Minuten. Eine Kepler-Langzeitkadenz von 29.4 Minuten verschmiert den Eintritt also fast vollständig. Deshalb nutzt der Fit unten die 1-Minuten-Kadenz, und das Modell mittelt jede Belichtung über ihre Dauer.

**4. Quadratische Randverdunkelung.** Der Sternrand ist dunkler als die Mitte, weil man dort in kühlere, höhere Schichten blickt. Das quadratische Gesetz beschreibt die Intensität als
$$I(\mu) = I(1)\left[1 - u_1(1-\mu) - u_2(1-\mu)^2\right], \qquad \mu = \cos\theta = \sqrt{1 - (r/R_\star)^2} .$$
Die Parametrisierung von Kipping (2013), $q_1 = (u_1 + u_2)^2$ und $q_2 = u_1 / \left(2(u_1 + u_2)\right)$, deckt mit $q_1, q_2 \in [0, 1]$ genau die physikalisch sinnvollen Profile ab (überall positiv, zum Rand hin abnehmend). Umgekehrt gilt $u_1 = 2\sqrt{q_1}\,q_2$ und $u_2 = \sqrt{q_1}\,(1 - 2q_2)$.

**5. Mittlere Sterndichte aus Keplers drittem Gesetz.** Aus $a^3/P^2 = G(M_\star + M_p)/(4\pi^2) \approx GM_\star/(4\pi^2)$ folgt nach Division durch $R_\star^3$
$$\rho_\star = \frac{3M_\star}{4\pi R_\star^3} = \frac{3\pi}{G P^2}\left(\frac{a}{R_\star}\right)^3 .$$
Weil $a/R_\star$ und $P$ allein aus der Lichtkurve folgen, misst die Transitanalyse die mittlere Dichte des Sterns, ohne seine Masse oder seinen Radius zu kennen (Seager & Mallén-Ornelas 2003).

**6. Das BLS-Kriterium.** Für Messwerte $f_i$ mit Gewichten $w_i = 1/\sigma_i^2$ (normiert auf $\sum_i w_i = 1$, Mittelwert abgezogen) testet BLS eine Kastenfunktion, die den Anteil $r = \sum_{i\in\text{in}} w_i$ der Gewichte umfasst. Die beste Kastentiefe ergibt sich aus der Summe $s = \sum_{i\in\text{in}} w_i f_i$, und die Verbesserung von $\chi^2$ gegenüber einer Konstanten ist
$$\Delta\chi^2 = \frac{s^2}{r(1-r)} .$$
BLS maximiert diese Größe über Periode, Phase und Dauer (Kovács, Zucker & Mazeh 2002). Ist das Periodengitter zu grob, trifft es den echten Peak nicht und landet auf einem Alias wie $P/2$ oder $P/3$, bei dem ebenfalls Kästen auf die Transits fallen.

#### **Daten**

Die Daten stammen aus der Kepler-Hauptmission (MAST). Ziel ist **Kepler-8 b**, ein aufgeblähter heißer Jupiter (entdeckt 2010) um einen späten F-Stern, mit einer Umlaufzeit von 3.52 Tagen. Ein anderer Planet lässt sich über `SELECTED_PLANET` wählen, solange es für ihn Kepler-Daten gibt.

Für die Suche dient ein festes Quartal (`QUARTER`) in Langzeitkadenz. Ohne diese Festlegung liefert MAST die Suchergebnisse in wechselnder Reihenfolge, und das Ergebnis hinge vom Zufall ab.

In [ ]:
# In Colab fehlen lightkurve und astroquery; lokal sind sie meist schon installiert
import importlib.util, subprocess, sys
missing = [p for p in ("lightkurve", "astroquery") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [ ]:
import os
from pathlib import Path

# Downloads landen im Ordner "astrodata" neben dem Notebook statt in den Caches im Home-Verzeichnis.
# Das muss vor dem ersten Import von astropy passieren; sonst den Kernel neu starten.
ASTRODATA = Path.cwd() / "astrodata"
for sub in ("astropy", "lightkurve"):
    (ASTRODATA / sub).mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(ASTRODATA)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
from scipy.optimize import curve_fit

SELECTED_PLANET = "Kepler-8 b"   # Planet mit Kepler-Daten
QUARTER = 1                      # Kepler-Quartal für die BLS-Suche (Langzeitkadenz, 29.4 min)
G_CGS = 6.674e-8                 # Gravitationskonstante [cm³ g⁻¹ s⁻²]

cat = NasaExoplanetArchive.query_criteria(
    table="pscomppars", where=f"pl_name='{SELECTED_PLANET}'",
    select="pl_name, hostname, pl_orbper, pl_rade, pl_ratror, pl_ratdor, pl_imppar, pl_orbincl, pl_trandur, "
           "st_rad, st_mass, st_dens, st_teff").to_pandas().iloc[0]

print(f"Planet:          {cat['pl_name']}  (Stern {cat['hostname']}, T_eff = {cat['st_teff']:.0f} K)")
print(f"Katalog-Periode: {cat['pl_orbper']:.5f} d")
print(f"Katalog-Radius:  {cat['pl_rade']:.2f} R_Erde  (Sternradius {cat['st_rad']:.2f} R_Sonne)")
print(f"Transitdauer:    {cat['pl_trandur']:.2f} h")

Wie sehen die Rohdaten aus, bevor irgendetwas korrigiert wird?

Das Target Pixel File enthält für jede Belichtung ein kleines Bild um den Stern. Die rot schraffierten Pixel bilden die Apertur der Kepler-Pipeline; ihre Summe ergibt die Rohlichtkurve.

In [ ]:
search = lk.search_targetpixelfile(SELECTED_PLANET, author="Kepler", exptime="long", quarter=QUARTER)
if len(search) == 0:
    raise RuntimeError(f"Keine Kepler-Pixeldaten für {SELECTED_PLANET} in Quartal {QUARTER}.")
tpf = search[0].download(quality_bitmask="default")
lc_raw = tpf.to_lightcurve(aperture_mask=tpf.pipeline_mask)

print(f"{search.table['mission'][0]}: {tpf.shape[0]} Belichtungen, {tpf.shape[1]} x {tpf.shape[2]} Pixel, "
      f"Dauer {np.ptp(tpf.time.value):.1f} Tage")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 2.2]})
tpf.plot(ax=axes[0], aperture_mask=tpf.pipeline_mask, scale="log", title=f"{SELECTED_PLANET}: Pixelbild (log)")
axes[1].plot(lc_raw.time.value, lc_raw.flux.value, ".", ms=2, c="black")
axes[1].set_xlabel("Zeit [BKJD = BJD − 2454833, Tage]")
axes[1].set_ylabel("Fluss in der Apertur [e⁻/s]")
axes[1].set_title(f"Rohlichtkurve, {search.table['mission'][0]}")
fig.tight_layout()
plt.show()

#### **Reduktion**

##### *1. Normieren und Trend entfernen*

Wie trennt man langsame Drifts des Teleskops von den kurzen Einbrüchen der Transits?

Die Lichtkurve wird auf 1 normiert, Ausreißer über 5σ werden entfernt, und ein Savitzky-Golay-Filter (`flatten`) schätzt den langsamen Trend. Das Fenster von 101 Belichtungen (etwa 50 Stunden) ist viel länger als ein Transit (3 Stunden) und kürzer als der Abstand der Transits (3.5 Tage).

In [ ]:
lc = lc_raw.remove_nans().remove_outliers(sigma=5).normalize()
lc_flat = lc.flatten(window_length=101)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(lc.time.value, lc.flux.value, ".", ms=2, c="grey", label="normiert")
axes[0].set_ylabel("Relativer Fluss")
axes[0].set_title(f"{SELECTED_PLANET}: Lichtkurve vor und nach dem Entfernen des Trends")
axes[0].legend()
axes[1].plot(lc_flat.time.value, lc_flat.flux.value, ".", ms=2, c="tab:green", label="ohne Trend")
axes[1].set_xlabel("Zeit [BKJD, Tage]")
axes[1].set_ylabel("Relativer Fluss")
axes[1].legend()
fig.tight_layout()
plt.show()

#### **Eigene Messung**

##### *2. Das BLS-Periodogramm*

Bei welcher Periode zeigt die Lichtkurve das stärkste periodische Kastensignal?

Die Suche läuft zuerst grob über 0.5 bis 20 Tage und dann fein um den stärksten Peak. Das Gitter ist dicht (`frequency_factor=1`, rund 50 000 Perioden). Ein zu grobes Gitter überspringt den schmalen echten Peak (siehe Variationen).

In [ ]:
DURATIONS = np.linspace(0.04, 0.30, 12)          # Test-Dauern [Tage]; BLS erwartet einfache Zahlen ohne Einheiten

pg_coarse = lc_flat.to_periodogram(method="bls", minimum_period=0.5, maximum_period=20.0,
                                   duration=DURATIONS, frequency_factor=1)
p_coarse = pg_coarse.period_at_max_power.value
pg = lc_flat.to_periodogram(method="bls", period=np.linspace(0.995 * p_coarse, 1.005 * p_coarse, 4000),
                            duration=DURATIONS)

period = pg.period_at_max_power.value              # [d]
t0 = pg.transit_time_at_max_power.value            # [BKJD]
duration = pg.duration_at_max_power.value          # [d]
depth = float(pg.depth_at_max_power.value)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(pg_coarse.period.value, pg_coarse.power.value, c="black", lw=0.6)
ax.axvline(period, color="tab:red", ls="--", alpha=0.7, label=f"Maximum: P = {period:.4f} d")
for n in (2, 3):
    ax.axvline(period / n, color="grey", ls=":", lw=1, label="Aliase P/2, P/3" if n == 2 else None)
ax.set_xscale("log")
ax.set_xlabel("Periode [Tage]")
ax.set_ylabel("BLS-Power")
ax.set_title(f"BLS-Periodogramm, {SELECTED_PLANET}, Quartal {QUARTER} ({len(pg_coarse.period)} Perioden)")
ax.legend()
fig.tight_layout()
plt.show()

print(f"Periode:        {period:.5f} d   (Katalog {cat['pl_orbper']:.5f} d, Abweichung "
      f"{abs(period - cat['pl_orbper']) / cat['pl_orbper'] * 100:.3f} %)")
print(f"Transitmitte:   T0 = {t0:.4f} BKJD")
print(f"Kastendauer:    {duration * 24:.2f} h   (Katalog T14 = {cat['pl_trandur']:.2f} h)")
print(f"Kastentiefe:    {depth * 1e6:.0f} ppm  ->  Rp/R* = {np.sqrt(depth):.4f}   (Katalog {cat['pl_ratror']:.4f})")

##### *3. Gefaltete Lichtkurve*

Wie sieht der Transit aus, wenn alle Transits des Quartals auf die gefundene Periode gefaltet werden?

In [ ]:
folded = lc_flat.fold(period=period, epoch_time=t0)
binned = folded.bin(time_bin_size=duration / 8)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(folded.time.value * 24, folded.flux.value, ".", ms=2, c="lightgrey", label="Belichtungen")
ax.plot(binned.time.value * 24, binned.flux.value, "o-", ms=3, c="tab:red", label="gemittelt")
ax.set_xlim(-3 * duration * 24, 3 * duration * 24)
ax.set_xlabel("Zeit seit Transitmitte [h]")
ax.set_ylabel("Relativer Fluss")
ax.set_title(f"{SELECTED_PLANET}: gefaltet mit P = {period:.4f} d ({int(np.ptp(lc_flat.time.value) / period)} Transits)")
ax.legend()
fig.tight_layout()
plt.show()

#### **Modell und Fit**

##### *4. Physikalisches Transitmodell mit Randverdunkelung*

Welches Radiusverhältnis $R_p/R_\star$, welcher Stoßparameter $b$ und welche Sterndichte $\rho_\star$ passen zur Form des Transits?

Der BLS-Kasten kennt nur Periode, Epoche, Dauer und eine mittlere Tiefe. Für die Form braucht man ein physikalisches Modell. Die Zelle

1. lädt sechs Monate Pipeline-Lichtkurven (PDCSAP-Fluss) in 1-Minuten-Kadenz, damit Ein- und Austritt aufgelöst sind (Herleitung, Punkt 3);
2. verfeinert die Ephemeride mit BLS auf einem engen Gitter um die Periode aus Schritt 2;
3. entfernt den Trend mit einem gleitenden Median, bei dem die Transits ausgeblendet sind, faltet und mittelt in 2-Minuten-Intervallen;
4. passt das Modell mit kleinsten Quadraten an, gestartet von vier Stoßparametern. $b$, $a/R_\star$ und die Randverdunkelung hängen eng zusammen; ein einzelner Start blieb in einem Nebenminimum hängen, mit systematischen Residuen bei Ein- und Austritt.

Das Modell integriert die randverdunkelte Sternscheibe numerisch über Ringe und berechnet für jeden Ring den Teil, den die Planetenscheibe verdeckt. Es nimmt eine Kreisbahn an.

In [ ]:
N_FILES = 6            # Lichtkurven-Dateien; bei Kepler-Kurzzeitkadenz ist eine Datei ein Monat
SHORT_CADENCE = True   # 1-Minuten-Kadenz löst Ein- und Austritt auf

def transit_flux(z, p, u1, u2, n_r=400):
    """Relativer Fluss eines quadratisch randverdunkelten Sterns, den ein Planet mit Radius p (in Sternradien)
    im projizierten Abstand z bedeckt. Numerische Integration über Ringe der Sternscheibe (Genauigkeit ~1e-6)."""
    r = (np.arange(n_r) + 0.5) / n_r                         # Ringradien auf der Sternscheibe
    mu = np.sqrt(1 - r ** 2)
    I = 1 - u1 * (1 - mu) - u2 * (1 - mu) ** 2               # Intensität mit Randverdunkelung
    z = np.atleast_1d(z)[:, None]
    # halber Winkel des Bogens von Ring r, der innerhalb der Planetenscheibe liegt
    cos_th = np.clip((r ** 2 + z ** 2 - p ** 2) / (2 * r * np.maximum(z, 1e-12)), -1, 1)
    theta = np.where(r + z <= p, np.pi, np.where(np.abs(r - z) >= p, 0.0, np.arccos(cos_th)))
    return 1 - (I * theta * r).sum(axis=1) / (np.pi * (I * r).sum())

def transit_model(t, dt0, p, a_rs, b, q1, q2, period_d, exptime_d, n_sub):
    """Kreisbahn; t in Tagen seit Transitmitte; Randverdunkelung nach Kipping (2013) mit q1, q2 in [0, 1].
    Jede Belichtung wird über ihre Dauer gemittelt (n_sub Stützpunkte)."""
    u1, u2 = 2 * np.sqrt(q1) * q2, np.sqrt(q1) * (1 - 2 * q2)
    inc = np.arccos(np.clip(b / a_rs, 0, 1))
    offsets = (np.arange(n_sub) - (n_sub - 1) / 2) / n_sub * exptime_d
    phase = 2 * np.pi * (np.asarray(t)[:, None] + offsets - dt0).ravel() / period_d
    z = a_rs * np.sqrt(np.sin(phase) ** 2 + (np.cos(inc) * np.cos(phase)) ** 2)
    z = np.where(np.cos(phase) > 0, z, 10.0)                  # Planet hinter dem Stern: kein Transit
    return transit_flux(z, p, u1, u2).reshape(len(t), n_sub).mean(axis=1)

def running_median(t, f, window_d, use):
    """Trend = gleitender Median über die Punkte in `use` (Fenster in Tagen), auf alle Zeiten interpoliert."""
    s = pd.Series(f[use], index=pd.to_datetime(t[use], unit="D"))
    trend = s.rolling(pd.Timedelta(days=window_d), center=True, min_periods=5).median().to_numpy()
    return np.interp(t, t[use], trend)

def fold_phase(t, P, t0):
    return (t - t0 + 0.5 * P) % P - 0.5 * P                  # Tage seit der nächsten Transitmitte

# Daten: Pipeline-Lichtkurven, feste Reihenfolge (Q00, Q01, ...), damit jeder Lauf dieselben Dateien nimmt
lc_search = lk.search_lightcurve(SELECTED_PLANET, author="Kepler")
exptimes = np.asarray(lc_search.table["exptime"], dtype=float)
FIT_EXPTIME = exptimes.min() if SHORT_CADENCE and exptimes.min() <= 120 else 1800.0
lc_search = lc_search[exptimes == FIT_EXPTIME]
lc_search = lc_search[np.argsort(np.asarray(lc_search.table["mission"]), kind="stable")]
n_files = min(N_FILES, len(lc_search))
print(f"Lade {n_files} von {len(lc_search)} Lichtkurven ({FIT_EXPTIME:.0f} s Kadenz) ...")
stitched = lc_search[:n_files].download_all(quality_bitmask="default").stitch()    # jede Datei auf 1 normiert
unmask = lambda q: np.asarray(getattr(q, "unmasked", q).value, dtype=float)       # einfache Arrays
t_all, f_all, e_all = unmask(stitched.time), unmask(stitched.flux), unmask(stitched.flux_err)
good = np.isfinite(t_all) & np.isfinite(f_all) & np.isfinite(e_all)
order = np.argsort(t_all[good])
t_all, f_all, e_all = t_all[good][order], f_all[good][order], e_all[good][order]
print(f"{len(t_all)} Belichtungen über {np.ptp(t_all):.0f} Tage")

# Ephemeride: BLS auf engem Gitter um die Periode aus Schritt 2
T14_guess = cat["pl_trandur"] / 24 if np.isfinite(cat["pl_trandur"]) else duration
f_rough = f_all / running_median(t_all, f_all, 1.0, np.ones(len(t_all), bool))
bls = BoxLeastSquares(t_all, f_rough, e_all).power(np.linspace(0.998 * period, 1.002 * period, 3000),
                                                   T14_guess * np.array([0.6, 0.8, 1.0, 1.2]), objective="snr")
k = np.argmax(bls.power)
P_d, t0_d, T14, depth_bls = bls.period[k], bls.transit_time[k], bls.duration[k], bls.depth[k]

# Trend mit ausgeblendeten Transits, dann falten
out_of_transit = np.abs(fold_phase(t_all, P_d, t0_d)) > 0.75 * T14
f_det = f_all / running_median(t_all, f_all, max(3 * T14, 0.75), out_of_transit)
ph = fold_phase(t_all, P_d, t0_d)
win = np.abs(ph) < 2.5 * T14
t_fit, f_fit = ph[win], f_det[win]

# Mittelwerte in 2-Minuten-Intervallen der Phase: einige hundert statt zehntausende Punkte
BIN_D = 2 / 1440
edges_fit = np.arange(-2.5 * T14, 2.5 * T14 + BIN_D, BIN_D)
ib = np.digitize(t_fit, edges_fit) - 1
nbins = len(edges_fit) - 1
n_in = np.bincount(ib, minlength=nbins)[:nbins]
s1 = np.bincount(ib, weights=f_fit, minlength=nbins)[:nbins]
s2 = np.bincount(ib, weights=f_fit ** 2, minlength=nbins)[:nbins]
ok = n_in >= 3
tb = (0.5 * (edges_fit[1:] + edges_fit[:-1]))[ok]
fb = s1[ok] / n_in[ok]
eb = np.sqrt(np.maximum(s2[ok] / n_in[ok] - fb ** 2, 1e-12) / (n_in[ok] - 1))    # Standardfehler des Mittelwerts
EXPTIME_D = FIT_EXPTIME / 86400
N_SUB = 1 if FIT_EXPTIME <= 120 else 7                    # Mittelung über die Belichtung nur bei Langzeitkadenz nötig

model = lambda t, dt0, p, a_rs, b, q1, q2: transit_model(t, dt0, p, a_rs, b, q1, q2, P_d, EXPTIME_D, N_SUB)
bounds = ([-0.02, 0.01, 1.5, 0.0, 0.0, 0.0], [0.02, 0.5, 50.0, 1.2, 1.0, 1.0])
# Mehrere Starts in b; a/R* passend zur Dauer, T14 ≈ P/π · sqrt((1+p)² − b²) / (a/R*)
p_start = np.sqrt(max(depth_bls, 1e-5))
best = None
for b_start in [0.1, 0.4, 0.6, 0.8]:
    a_start = np.clip(P_d / (np.pi * T14_guess) * np.sqrt((1 + p_start) ** 2 - b_start ** 2), 2, 40)
    try:
        popt_i, pcov_i = curve_fit(model, tb, fb, p0=[0.0, p_start, a_start, b_start, 0.36, 0.3], sigma=eb,
                                   absolute_sigma=True, bounds=bounds, maxfev=20000)
    except RuntimeError:
        continue
    chi2_i = np.sum(((fb - model(tb, *popt_i)) / eb) ** 2)
    print(f"  Start b = {b_start:.1f}:  χ² = {chi2_i:8.1f},  b = {popt_i[3]:.3f},  Rp/R* = {popt_i[1]:.4f}")
    if best is None or chi2_i < best[0]:
        best = (chi2_i, popt_i, pcov_i)
_, popt, pcov = best
perr = np.sqrt(np.diag(pcov))
dt0_fit, p_fit, a_fit, b_fit, q1_fit, q2_fit = popt
inc_fit = np.degrees(np.arccos(b_fit / a_fit))
chi2_red = np.sum(((fb - model(tb, *popt)) / eb) ** 2) / (len(tb) - len(popt))
T14_fit = P_d / np.pi * np.arcsin(np.sqrt((1 + p_fit) ** 2 - b_fit ** 2) / (a_fit * np.sin(np.radians(inc_fit))))
rho_fit = 3 * np.pi * a_fit ** 3 / (G_CGS * (P_d * 86400) ** 2)      # Herleitung, Punkt 5
rho_err = rho_fit * 3 * perr[2] / a_fit

# Darstellung: alle Belichtungen, 60 Mittelwerte, Modell und BLS-Kasten
tt = np.linspace(-2.5 * T14, 2.5 * T14, 800)
edges = np.linspace(-2.5 * T14, 2.5 * T14, 61)
idx = np.digitize(t_fit, edges) - 1
t_bin = 0.5 * (edges[1:] + edges[:-1])
f_bin = np.array([np.mean(f_fit[idx == i]) if np.any(idx == i) else np.nan for i in range(60)])
e_bin = np.array([np.std(f_fit[idx == i]) / np.sqrt(max((idx == i).sum(), 1)) for i in range(60)])

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
sub = np.random.default_rng(0).choice(len(t_fit), min(len(t_fit), 20000), replace=False)
axes[0].plot(t_fit[sub] * 24, f_fit[sub], ".", ms=1, c="lightgrey",
             label=f"Belichtungen ({n_files} Dateien, {int(np.ptp(t_all) / P_d)} Umläufe)")
axes[0].errorbar(t_bin * 24, f_bin, e_bin, fmt="o", c="black", ms=3, label="gemittelt")
axes[0].plot(tt * 24, model(tt, *popt), c="tab:red", lw=2, label=f"Transitmodell: Rp/R* = {p_fit:.4f}, b = {b_fit:.2f}")
axes[0].plot(tt * 24, 1 - depth_bls * (np.abs(tt) < T14 / 2), c="tab:blue", lw=1, ls="--", label="BLS-Kasten")
axes[0].set_ylabel("Relativer Fluss")
axes[0].set_title(f"{SELECTED_PLANET}: Transitmodell ({FIT_EXPTIME:.0f} s Kadenz)")
axes[0].legend(fontsize=8)
axes[1].errorbar(tb * 24, (fb - model(tb, *popt)) * 1e6, eb * 1e6, fmt=".", c="grey", ms=2, label="2-min-Mittel")
axes[1].axhline(0, color="tab:red", lw=1)
axes[1].set_xlabel("Zeit seit Transitmitte [h]")
axes[1].set_ylabel("Residuum [ppm]")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

u1_fit, u2_fit = 2 * np.sqrt(q1_fit) * q2_fit, np.sqrt(q1_fit) * (1 - 2 * q2_fit)
print(f"Ephemeride:  P = {P_d:.5f} d,  T0 = {t0_d + dt0_fit:.4f} BKJD")
print(f"Fit: reduziertes χ² = {chi2_red:.2f};  Randverdunkelung u1 = {u1_fit:.2f}, u2 = {u2_fit:.2f}")

#### **Eigene Messung gegen den Katalog**

##### *5. Vergleich mit dem NASA Exoplanet Archive*

Wie gut stimmen die eigenen Werte mit den veröffentlichten überein, und woher kommen die Unterschiede?

In [ ]:
rho_mr = cat["st_mass"] * 1.989e33 / (4 / 3 * np.pi * (cat["st_rad"] * 6.957e10) ** 3)   # aus Katalog-M*, -R*
rows = [
    ("Periode P [d]",               f"{P_d:.5f}",                               f"{cat['pl_orbper']:.5f}"),
    ("Radiusverhältnis Rp/R*",      f"{p_fit:.4f} ± {perr[1]:.4f}",             f"{cat['pl_ratror']:.4f}"),
    ("  (BLS-Kasten)",              f"{np.sqrt(depth_bls):.4f}",                ""),
    ("Skalierte Halbachse a/R*",    f"{a_fit:.2f} ± {perr[2]:.2f}",             f"{cat['pl_ratdor']:.2f}"),
    ("Stoßparameter b",             f"{b_fit:.3f} ± {perr[3]:.3f}",             f"{cat['pl_imppar']:.3f}"),
    ("Bahnneigung i [°]",           f"{inc_fit:.2f}",                           f"{cat['pl_orbincl']:.2f}"),
    ("Transitdauer T14 [h]",        f"{T14_fit * 24:.2f}",                      f"{cat['pl_trandur']:.2f}"),
    ("Planetenradius [R_Erde]",     f"{p_fit * cat['st_rad'] * 109.08:.2f}",   f"{cat['pl_rade']:.2f}"),
    ("Sterndichte ρ* [g/cm³]",      f"{rho_fit:.3f} ± {rho_err:.3f}",           f"{cat['st_dens']:.3f} (st_dens)"),
    ("  aus Katalog-M*, -R*",       "",                                         f"{rho_mr:.3f}"),
]
print(f"{'Größe':<28}{'eigene Messung':<22}{'Katalog'}")
print("-" * 68)
for name, own, ref in rows:
    print(f"{name:<28}{own:<22}{ref}")
print("\nFehler: nur statistisch, aus der Kovarianzmatrix des Fits. Der Planetenradius nutzt den Katalog-Sternradius "
      f"{cat['st_rad']:.2f} R_Sonne. Sonne zum Vergleich: ρ = 1.41 g/cm³.")

Ergebnis beim Lauf vom 2026-10-12 (Kepler-8 b, Quartal 1, sechs Monate 1-Minuten-Kadenz):

- **Periode:** BLS findet auf einem einzigen Quartal (9 Transits) 3.52235 d, 0.004 % neben dem Katalogwert; die Verfeinerung auf sechs Monaten gibt 3.52244 d.
- **Radiusverhältnis:** $R_p/R_\star = 0.0969$ gegen 0.0958 im Katalog, Planetenradius 15.7 statt 15.9 $R_\oplus$. Der BLS-Kasten unterschätzt es (0.091), weil er die Randverdunkelung und den endlichen Eintritt nicht kennt: Er mittelt über den gekrümmten Boden und die Flanken.
- **Geometrie:** $b = 0.76$ und $a/R_\star = 6.53$ gegen 0.72 und 6.85. Die drei Größen $b$, $a/R_\star$ und die Randverdunkelung können sich gegenseitig ausgleichen: Ein etwas größeres $b$ mit kleinerem $a/R_\star$ gibt fast dieselbe Kurve. Die angegebenen Fehler (±0.001 in $b$) sind nur statistisch und deshalb viel zu klein.
- **Sterndichte:** 0.42 g/cm³ gegen 0.65 (`st_dens`) und 0.52 g/cm³ (aus Katalog-Masse und -Radius). Weil $\rho_\star \propto (a/R_\star)^3$, wird ein Unterschied von 5 % in $a/R_\star$ zu 15 % in der Dichte. Die Katalogtabelle `pscomppars` mischt außerdem Quellen: Ihr `st_dens` passt nicht zu ihrer eigenen Sternmasse und ihrem Sternradius.
- **Fitgüte:** reduziertes $\chi^2 = 1.19$, die Residuen zeigen keine Struktur bei Ein- und Austritt. Erst der Start bei $b = 0.8$ fand das beste Minimum; die Starts bei 0.1 und 0.4 blieben bei deutlich schlechterem $\chi^2$ hängen.

#### **Variationen**

Jeweils zuerst die Vorhersage notieren, dann den Code ändern und prüfen:

1. **Zu grobes Periodengitter:** Setze in Schritt 2 `frequency_factor=500` und `QUARTER = 17`. Das Gitter enthält dann nur rund 100 Perioden. *(Vorhersage: Der echte Peak ist schmaler als der Gitterabstand; BLS landet auf einem Alias, bei Quartal 17 auf $P/3 \approx 1.17$ d.)*
2. **Kurzes Trendfenster:** Verkleinere in Schritt 1 `window_length` von 101 auf 11 (etwa 5 Stunden). Was passiert mit der Transittiefe? *(Vorhersage: Das Filter hält die Flanken des Transits für Trend und macht den Einbruch flacher.)*
3. **Ohne Randverdunkelung:** Halte im Fit $q_1 = 0$ fest (gleichmäßig helle Scheibe). Wie sehen die Residuen bei Ein- und Austritt und in der Transitmitte aus?
4. **Langzeitkadenz:** Setze `SHORT_CADENCE = False` und vergleiche $b$ und $a/R_\star$ mit und ohne Mittelung über die Belichtung (`N_SUB = 7` gegen `N_SUB = 1`). *(Vorhersage: Ohne Mittelung wirkt der verschmierte Eintritt wie ein streifender Transit, $b$ wird zu groß.)*

#### **Nächste Schritte**

So würde eine Analyse in der Forschung weitergehen:
- **MCMC statt kleinster Quadrate:** eine bayessche Anpassung (zum Beispiel mit `emcee`) mit Vorwissen über die Randverdunkelung aus Sternatmosphären-Modellen (`ldtk` oder Tabellen von Claret). Erst damit werden die Unsicherheiten von $b$, $a/R_\star$ und $\rho_\star$ realistisch, weil diese Größen stark zusammenhängen.
- **Transit Timing Variations:** einzelne Transitzeiten jedes Umlaufs messen; Abweichungen von der strengen Periode verraten weitere Planeten im System.
- **Transit Least Squares (TLS):** sucht mit einer realistischen Transitform statt eines Kastens und findet kleine Planeten empfindlicher.
- **Vetting:** prüfen, ob der Transit wirklich vom Zielstern kommt (Notebook `exoplanets_vetting.ipynb`).

#### **Referenzen**

- Kovács, G., Zucker, S. & Mazeh, T. (2002): *A box-fitting algorithm in the search for periodic transits*, A&A 391, 369.
- Mandel, K. & Agol, E. (2002): *Analytic Light Curves for Planetary Transit Searches*, ApJ 580, L171.
- Seager, S. & Mallén-Ornelas, G. (2003): *A Unique Solution of Planet and Star Parameters from an Extrasolar Planet Transit Light Curve*, ApJ 585, 1038.
- Winn, J. N. (2010): *Transits and Occultations*, in *Exoplanets*, ed. S. Seager, arXiv:1001.2010.
- Kipping, D. M. (2013): *Efficient, uninformative sampling of limb darkening coefficients for two-parameter laws*, MNRAS 435, 2152.
- Jenkins, J. M. et al. (2010): *Overview of the Kepler Science Processing Pipeline*, ApJ 713, L87.